# NB2 · Verinin hazırlanması

**Üretken Yapay Zekâ Araçları ile Klinik Karar Destek Sistemleri Geliştirilmesi**  
Sağlık Bilimlerinde Teknoloji ve Yapay Zekâ Okuryazarlığı Eğitimi · Akdeniz Üniversitesi · 18 Eylül 2026

Prof. Dr. Utku Köse · Süleyman Demirel Üniversitesi, Bilgisayar Mühendisliği Bölümü  
Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM) Müdürü · utkukose@sdu.edu.tr

---


## Defterin kullanımı

Bu defterde kodu siz yazmayacaksınız. Her adımda bir istem verilmektedir. İstemi
kopyalayarak bir üretken yapay zekâ aracına aktarınız, aracın ürettiği kodu defterdeki
boş hücreye yapıştırınız ve çalıştırınız.

Her yapıştırma hücresinin ardından bir kontrol hücresi yer alır. Kontrol hücresi,
ürettiğiniz kodun beklenen çıktıyı verip vermediğini sınar ve eksik bulunması hâlinde
bunu bildirir. Kontrol hücreleri bu defterin sabit parçasıdır; değiştirilmemelidir.

İstemlerin sonunda KABUL ÖLÇÜTLERİ başlıklı bir bölüm bulunur. Kabul ölçütleri, kodun hangi adla
hangi değişkeni üretmesi gerektiğini belirler ve kontrol hücresi bu tanımı esas alır.
Bir işi üretken yapay zekâ aracına devrederken arayüzü tanımlamak kullanıcının
sorumluluğundadır. Kabul ölçütü içermeyen bir istem, denetlenmesi mümkün olmayan bir kod
üretir.

Kontrol hücresi eksik bildirdiğinde üretilen kodu yapay zekâ aracına geri veriniz,
bildirilen eksikleri aktarınız ve kodu yeniden ürettiriniz. Bu döngü olağandır; ilk
denemede kabul ölçütlerinin tam olarak karşılanması beklenmez.


## Ele alınan problem

Yoğun bakıma kabul edilen bir hastanın üç günden uzun süre kalıp kalmayacağı
öngörülecektir. Sorunun klinik karşılığı yatak kapasitesi planlamasıdır. Uzun kalacak
hasta önceden belirlenebilirse kapasite buna göre düzenlenir; belirlenemediğinde
kapasite plansız dolar ve sıradaki hastanın kabulü gecikir.

Öngörü, hastanın yoğun bakıma girişinden altı saat sonra yapılacaktır. Bu nokta
sistemin karar anıdır. Karar anına kadar kaydedilmiş veriler kullanılabilir,
sonrasında kaydedilenler kullanılamaz. Söz konusu kural dersin temel kuralıdır ve
gerekçesi ikinci adımda ele alınacaktır.

Veri kaynağı olarak MIMIC-IV demo kümesi kullanılacaktır. Küme yüz hastadan oluşur,
açık erişimlidir ve kimlik doğrulaması gerektirmez. Dosyalar internetten doğrudan
okunacak, önceden indirme yapılmayacaktır.


## Hazırlık


In [ ]:
!pip -q install pandas numpy

import urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
urllib.request.urlretrieve(f'{REPO}/workshop/checks.py', 'checks.py')

import pandas as pd
import checks
checks.LANG = 'tr'

print('Kontrol modülü hazır.')


---

## Adım 1 · Tabloların getirilmesi

MIMIC verisi birbirine bağlı ayrı tablolarda tutulur. Hastanın yaşı ve cinsiyeti bir
tabloda, hastane yatışına ait bilgiler ikinci tabloda, yoğun bakım süresi üçüncü
tabloda yer alır. İlk adımda bu üç tablo getirilecektir.

İstemde dosya adresleri ve sütun adları açıkça verilmiştir. Üretken yapay zekâ
aracının veri şemasını bilmesi beklenmemelidir. Şema verilmediğinde araç eksik bilgiyi
tahminle tamamlar ve tahmin çoğunlukla hatalı olur. Veri sözlüğünü isteme eklemek
kullanıcının sorumluluğundadır.


### İstem 1

```
Google Colab ortamında çalışacak tek bir Python hücresi yaz.

Aşağıdaki üç dosyayı pandas ile doğrudan internetten oku. Üçü de gzip sıkıştırmalı
CSV dosyasıdır:

https://physionet.org/files/mimic-iv-demo/2.2/hosp/patients.csv.gz
https://physionet.org/files/mimic-iv-demo/2.2/hosp/admissions.csv.gz
https://physionet.org/files/mimic-iv-demo/2.2/icu/icustays.csv.gz

Sütun yapısı şöyledir:
patients:   subject_id, gender, anchor_age
admissions: subject_id, hadm_id, admission_type, insurance, race
icustays:   subject_id, hadm_id, stay_id, first_careunit, intime, outtime, los

intime ve outtime tarih sütunlarıdır, pandas tarih tipine çevir.
los, yoğun bakım yatış süresini gün cinsinden verir.

KABUL ÖLÇÜTLERİ
Kod tamamlandığında aşağıdaki üç değişken hazır olmalıdır:
  hastalar     -> patients tablosu
  yatislar     -> admissions tablosu
  yogun_bakim  -> icustays tablosu
Her tablonun satır sayısını ekrana yaz. Bunun dışında çıktı üretme.
```


In [ ]:
# Ürettiğiniz kodu bu hücreye yapıştırınız ve çalıştırınız.


### Kontrol 1


In [ ]:
checks.check_tables(hastalar=hastalar, yatislar=yatislar, yogun_bakim=yogun_bakim)


---

## Adım 2 · Karar anına gelmeyen yatışların ayıklanması

Yoğun bakım yatışlarının bir bölümü altı saat dolmadan sona erer. Bu hastalar sistemin
karar anına hiç ulaşmamıştır ve haklarında bir öngörü üretilmesi anlamsızdır. Söz
konusu yatışlar çalışma kümesinden çıkarılmalıdır.

Bu ayıklama küçük bir işlem gibi görünür, ancak kohort tanımının bir parçasıdır. Bir
klinik karar destek sisteminin hangi hastaları kapsadığı, hangi hastaları kapsam
dışında bıraktığı kadar önemlidir.


### İstem 2

```
yogun_bakim tablosunu kullanarak çalışma kümesini oluştur. Tek bir Python hücresi yaz.

los değeri 0.25 günden küçük olan satırları çıkar. Bu hastalar altı saat dolmadan
yoğun bakımdan çıkmıştır.

Kaç satırın çıkarıldığını ve geriye kaç satır kaldığını ekrana yaz.

KABUL ÖLÇÜTLERİ
Sonuç df adında bir DataFrame olmalıdır.
df içinde subject_id, hadm_id, stay_id, first_careunit, intime, outtime, los
sütunları bulunmalıdır.
```


In [ ]:
# Ürettiğiniz kodu bu hücreye yapıştırınız ve çalıştırınız.


### Kontrol 2


In [ ]:
checks.check_columns(
    df,
    required=['subject_id', 'hadm_id', 'stay_id', 'first_careunit',
              'intime', 'outtime', 'los'],
)

satir_sayisi_adim2 = len(df)
print(f'\nAdım 2 sonunda satır sayısı: {satir_sayisi_adim2}')


---

## Adım 3 · Hedef değişkenin üretilmesi

Hedef değişken los sütunundan türetilecektir. Üç günden uzun yatışlarda değer 1,
diğerlerinde 0 olacaktır.

Bu noktada dikkat edilmesi gereken bir husus bulunmaktadır. los değeri hasta yoğun
bakımdan çıktıktan sonra belli olur. Sütun modele girdi olarak verilirse model
kusursuza yakın bir başarım gösterir. Sistem hastanede devreye alındığında ise karar
anında bu sütun boş olduğu için hiçbir çıktı üretilemez.

Bu duruma hedef sızıntısı adı verilir. Üretken yapay zekâ ile yazılan klinik kodda en
sık rastlanan hatadır. Sessiz ilerler, hata iletisi üretmez ve başarım değerlerini
yükselttiği için ilk bakışta olumlu görünür.

Aynı gerekçe outtime sütunu için de geçerlidir. Çıkış saatini bilen bir model yatış
süresini de bilir.

Bu nedenle hedef değişken üretildikten sonra her iki sütun da tablodan çıkarılacaktır.
Sızıntı, sonradan fark edilmeye çalışılmak yerine istem aşamasında engellenmektedir.


### İstem 3

```
df tablosunda hedef değişkeni üret ve sızıntıya yol açan sütunları çıkar. Tek bir
Python hücresi yaz.

1. hedef adında yeni bir sütun oluştur. los değeri 3'ten büyükse 1, değilse 0 olsun.
2. Hedef üretildikten sonra los ve outtime sütunlarını df tablosundan çıkar.

Hedef değişkenin dağılımını ekrana yaz.

KABUL ÖLÇÜTLERİ
df içinde hedef adında bir sütun bulunmalı ve yalnızca 0 ile 1 değerlerini almalıdır.
df içinde los ve outtime sütunları BULUNMAMALIDIR.
df satır sayısı değişmemelidir.
```


In [ ]:
# Ürettiğiniz kodu bu hücreye yapıştırınız ve çalıştırınız.


### Kontrol 3

Bu adımda üç ayrı kontrol çalıştırılmaktadır: Hedef değişkenin yapısı, çıkarılması
gereken sütunların durumu ve satır sayısının korunup korunmadığı.


In [ ]:
checks.check_target(df, target='hedef')


In [ ]:
checks.check_columns(df, required=['hedef'], forbidden=['los', 'outtime'])


In [ ]:
checks.check_rows(df, before=satir_sayisi_adim2)


---

## Adım 4 · Demografik ve yatış bilgilerinin eklenmesi

Çalışma kümesi şu ana kadar yalnızca yoğun bakım tablosundan gelen bilgileri
içermektedir. Yaş, cinsiyet ve yatış türü gibi bilgiler diğer iki tabloda yer alır ve
bunlar karar anında hastanın dosyasında mevcuttur.

Birleştirme işleminde gözden kaçan bir hata biçimi vardır. Birleştirme anahtarı doğru
seçilmediğinde satırlar çoğalır ve aynı yatış tabloda birden fazla kez görünür. Hata
iletisi üretilmez; yalnızca satır sayısı artar. Bu nedenle birleştirmenin ardından
satır sayısı ayrıca denetlenecektir.


### İstem 4

```
df tablosuna hasta ve yatış bilgilerini ekle. Tek bir Python hücresi yaz.

1. hastalar tablosundan gender ve anchor_age sütunlarını subject_id üzerinden ekle.
2. yatislar tablosundan admission_type ve insurance sütunlarını hadm_id üzerinden ekle.

Birleştirmeleri sol birleştirme olarak yap. df tablosundaki satır sayısı
değişmemelidir.

Birleştirme öncesi ve sonrası satır sayısını ekrana yaz.

KABUL ÖLÇÜTLERİ
df içinde şu sütunlar bulunmalıdır:
  subject_id, stay_id, hedef, gender, anchor_age, admission_type,
  insurance, first_careunit
df satır sayısı bir önceki adımdakiyle aynı kalmalıdır.
```


In [ ]:
# Ürettiğiniz kodu bu hücreye yapıştırınız ve çalıştırınız.


### Kontrol 4


In [ ]:
checks.check_columns(
    df,
    required=['subject_id', 'stay_id', 'hedef', 'gender', 'anchor_age',
              'admission_type', 'insurance', 'first_careunit'],
    forbidden=['los', 'outtime'],
)


In [ ]:
checks.check_rows(df, before=satir_sayisi_adim2)


Satır sayısı arttıysa birleştirme anahtarı hatalıdır. Bir hastanın birden fazla hastane
yatışı bulunabildiği için subject_id tek başına yatış düzeyinde bir anahtar değildir.
Böyle bir durumda istemi düzelterek kodu yeniden ürettiriniz.


---

## Adım 5 · Kohortun bütün olarak denetlenmesi

Adımlar tek tek denetlendi. Son olarak kohort bütün hâlinde sınanacaktır. Bu kontrol
önceki denetimleri tekrar eder ve bunlara iki inceleme ekler.

Birinci inceleme hasta kimliği üzerinedir. Bir hastanın birden fazla yoğun bakım yatışı
bulunabilir. Böyle bir durumda veri eğitim ve test olarak ayrılırken ayrımın hasta
düzeyinde yapılması gerekir. Aynı hastanın bir yatışı eğitim, diğeri test kümesine
düşerse model o hastayı tanır ve test başarımı olduğundan yüksek çıkar. Konu NB3'te
ele alınacaktır.

İkinci inceleme sütun adlarına dayalı bir sızıntı taramasıdır. Tarama yalnızca adlara
bakar ve güvence vermez. Asıl soruyu kullanıcı sormalıdır: Bu sütun karar anında dolu
muydu?


In [ ]:
checks.check_cohort(df, target='hedef', patient_id='subject_id')


---

## Adım 6 · Diğer veri türleri

Buraya kadar rutin hastane verisi kullanıldı. Söz konusu veri türü sayısal ve
kategorik sütunlardan oluşan bir tablodur. Katılımcıların bir bölümü ise görüntü,
fizyolojik zaman serisi veya klinik metin üzerinde çalışmaktadır.

Aşağıdaki istemler aynı işi farklı veri türleri için yapar. Kendi alanınıza uygun olanı
seçiniz. Bu noktadan sonraki bütün adımlar veri türünden bağımsız olarak aynı şekilde
ilerleyecektir.

Elinizde gerçek veri bulunmuyorsa istemler sentetik veri üretir. Sentetik veri ile
öğrenilen şey yöntemin kendisidir ve bu dersin amacı da yöntemin öğretilmesidir.


### İstem 6a · Tıbbi görüntü

```
MedMNIST koleksiyonundan probleme uygun bir alt küme yükleyen tek bir Python hücresi
yaz. Uygun koleksiyon bulunmuyorsa denetlenebilir sinyal içeren sentetik görüntü üret.

Problem tanımım: [buraya kendi probleminizi bir cümleyle yazınız]

Sınıf dağılımını ekrana yaz. Test kümesini, herhangi bir ön işleme adımı
öğrenilmeden önce ayır. Dört çarpı dörtlük bir örnek ızgarası göster.

KABUL ÖLÇÜTLERİ
X_train, X_test, y_train, y_test değişkenleri hazır olmalıdır.
y değerleri yalnızca 0 ve 1 olmalıdır.
```


### İstem 6b · Fizyolojik zaman serisi

```
Fizyolojik zaman serisi verisi hazırlayan tek bir Python hücresi yaz. Uygunsa
MIMIC-IV-ECG demo kümesini kullan, değilse sentetik sinyal üret.

Problem tanımım: [buraya kendi probleminizi bir cümleyle yazınız]

Örnekleme hızını ve pencere uzunluğunu ekrana yaz. Eğitim ve test ayrımını pencere
düzeyinde değil hasta düzeyinde yap; gerekçesini yorum satırı olarak ekle.

KABUL ÖLÇÜTLERİ
X_train, X_test, y_train, y_test değişkenleri hazır olmalıdır.
hasta_id_train ve hasta_id_test değişkenleri de bulunmalıdır.
```


### İstem 6c · Klinik metin

```
Sentetik klinik not derlemi üreten tek bir Python hücresi yaz.

Problem tanımım: [buraya kendi probleminizi bir cümleyle yazınız]
Notlar Türkçe olmalıdır.

Gerçek klinik metni güçleştiren kayıt özelliklerini ekle: Kısaltma, olumsuzlama,
belirsizlik ifadesi, şablon metin ve önceki nottan kopyalama. Etiket tek bir
anahtar kelimeden çıkarılabilir olmasın; hangi yüzeysel ipuçlarını bilerek
elediğini belirt.

KABUL ÖLÇÜTLERİ
df adında bir DataFrame olmalı, içinde metin ve hedef sütunları bulunmalıdır.
subject_id sütunu da bulunmalıdır.
Üç örnek notu etiketiyle birlikte yazdır.
```


### İstem 6d · Kendi tablo veriniz

```
Kendi problemim için sentetik bir hasta tablosu üreten tek bir Python hücresi yaz.

Problem tanımım: [buraya yazınız]
Tahmin etmek istediğim sonuç: [buraya yazınız]
Bu sonucun kendi hasta grubumdaki sıklığı: [yüzde olarak yazınız]
Karar anında elimde bulunan bilgiler: [sıralayınız]

Sonuç oranı belirttiğim yüzde olmalıdır. Gerçekçi oranda eksik veri ekle. Sonuç
gerçekleştikten sonra kaydedilecek hiçbir sütun bulunmasın.

KABUL ÖLÇÜTLERİ
df adında bir DataFrame olmalı, içinde hedef ve subject_id sütunları bulunmalıdır.
Verinin eğitim amaçlı sentetik veri olduğunu hücrenin başına yorum olarak yaz.
```


In [ ]:
# Kendi veri türünüz için ürettiğiniz kodu bu hücreye yapıştırınız.


### Kontrol 6

Tablo verisiyle devam edenler aşağıdaki hücreyi olduğu gibi çalıştırır. Görüntü veya
zaman serisi verisiyle çalışanlar bu kontrolü atlayabilir; onların sözleşmesi X ve y
değişkenleri üzerinedir.


In [ ]:
checks.check_cohort(df, target='hedef', patient_id='subject_id')


---

## Defter sonu · Veri raporu

Aşağıdaki hücre defterin sabit değerlendirme bölümüdür. Ürettiğiniz tablo üzerinde
çalışır ve NB3'e geçmeden önce bilinmesi gereken hususları bildirir.


In [ ]:
checks.data_report(df, target='hedef', patient_id='subject_id')


Raporun pozitif vaka sayısına ilişkin bir uyarı üretmesi beklenmektedir. Yüz hastalık
bir kümede kurulan modelin sonuçları yöntemi gösterir, başarımı kanıtlamaz. Ayrımın
sayısal karşılığı NB3'te ele alınacaktır.


## Bu defterde ele alınanlar

Kod yazmadan çalışan bir veri hattı kuruldu. Bu sırada üç uygulama benimsendi.

Veri sözlüğü isteme eklendi. Üretken yapay zekâ aracının şemayı bilmesi beklenmedi;
tablo adları, sütun adları ve dosya adresleri açıkça verildi.

Her isteme bir sözleşme eklendi. Böylece üretilen kodun beklenen çıktıyı verip
vermediği denetlenebildi ve denetim adım adım yapıldı.

Karar anı tanımlandı ve sonradan öğrenilen sütunlar çıkarıldı. Sızıntı, sonradan fark
edilmeye çalışılmak yerine istem aşamasında engellendi.

Kod üretme maliyeti düşmüştür; hangi kodun isteneceğini belirleme sorumluluğu kullanıcıda
kalır.

---

**Uyarı.** Bu defterde üretilen hiçbir çıktı doğrulanmış bir klinik araç değildir.
MIMIC-IV demo verisi tek bir Amerikan hastanesinden gelmektedir ve Türkiye'deki bir
yoğun bakım popülasyonunu temsil etmez. Materyal öğretim amaçlıdır.
